# Chess Elo Prediction — Improved XGBoost Model

**Key improvements over the baseline:**
- Added piece-type move rates (pawn, minor, major)
- Added castle timing (which move castling happened on)
- Added interaction/diff features between white and black
- Added opponent Elo as a feature (biggest R² boost)
- Added game complexity feature

**Why opponent Elo matters:** This dataset is all high-rated players (mean ~2247). Behavior stats alone barely differ between a 1900 and a 2400 player. But knowing your opponent's Elo is a strong prior — good players tend to face good players.

In [ ]:
%pip install python-chess xgboost scikit-learn pandas numpy matplotlib

In [ ]:
import chess.pgn
import chess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

## Step 1: Feature Extraction

In [ ]:
def extract_features(game):
    board = game.board()

    white_moves, black_moves = [], []
    white_captures = black_captures = 0
    white_checks   = black_checks   = 0
    white_castled  = black_castled  = False
    white_promotions = black_promotions = 0
    white_piece_diversity = set()
    black_piece_diversity = set()
    white_move_numbers, black_move_numbers = [], []

    # Piece-type move counters
    white_pawn_moves = black_pawn_moves = 0
    white_minor_moves = black_minor_moves = 0   # knight + bishop
    white_major_moves = black_major_moves = 0   # rook + queen

    move_number = 0
    for move in game.mainline_moves():
        san = board.san(move)
        is_white = board.turn
        piece = board.piece_at(move.from_square)
        piece_type = piece.piece_type if piece else 0

        if is_white:
            white_moves.append(san)
            white_piece_diversity.add(piece_type)
            white_move_numbers.append(move_number)
            if board.is_capture(move):                  white_captures += 1
            if move.promotion:                          white_promotions += 1
            if san.endswith("+") or san.endswith("#"): white_checks += 1
            if san in ("O-O", "O-O-O"):                white_castled = True
            if piece_type == chess.PAWN:                white_pawn_moves += 1
            if piece_type in (chess.KNIGHT, chess.BISHOP): white_minor_moves += 1
            if piece_type in (chess.ROOK,   chess.QUEEN):  white_major_moves += 1
        else:
            black_moves.append(san)
            black_piece_diversity.add(piece_type)
            black_move_numbers.append(move_number)
            if board.is_capture(move):                  black_captures += 1
            if move.promotion:                          black_promotions += 1
            if san.endswith("+") or san.endswith("#"): black_checks += 1
            if san in ("O-O", "O-O-O"):                black_castled = True
            if piece_type == chess.PAWN:                black_pawn_moves += 1
            if piece_type in (chess.KNIGHT, chess.BISHOP): black_minor_moves += 1
            if piece_type in (chess.ROOK,   chess.QUEEN):  black_major_moves += 1

        board.push(move)
        move_number += 1

    total_moves = move_number
    wc = len(white_moves) or 1
    bc = len(black_moves) or 1

    # On which half-move did castling happen? Earlier = more structured opening
    white_castle_move = next(
        (white_move_numbers[i] for i, m in enumerate(white_moves) if m in ("O-O","O-O-O")),
        total_moves
    )
    black_castle_move = next(
        (black_move_numbers[i] for i, m in enumerate(black_moves) if m in ("O-O","O-O-O")),
        total_moves
    )

    return {
        "total_moves":           total_moves,
        "result":                game.headers.get("Result", "*"),

        # White
        "white_move_count":      wc,
        "white_captures":        white_captures,
        "white_checks":          white_checks,
        "white_castled":         int(white_castled),
        "white_castle_move":     white_castle_move,
        "white_promotions":      white_promotions,
        "white_piece_diversity": len(white_piece_diversity),
        "white_capture_rate":    white_captures  / wc,
        "white_check_rate":      white_checks    / wc,
        "white_pawn_rate":       white_pawn_moves  / wc,
        "white_minor_rate":      white_minor_moves / wc,
        "white_major_rate":      white_major_moves / wc,
        "white_avg_move_number": np.mean(white_move_numbers) if white_move_numbers else 0,

        # Black
        "black_move_count":      bc,
        "black_captures":        black_captures,
        "black_checks":          black_checks,
        "black_castled":         int(black_castled),
        "black_castle_move":     black_castle_move,
        "black_promotions":      black_promotions,
        "black_piece_diversity": len(black_piece_diversity),
        "black_capture_rate":    black_captures  / bc,
        "black_check_rate":      black_checks    / bc,
        "black_pawn_rate":       black_pawn_moves  / bc,
        "black_minor_rate":      black_minor_moves / bc,
        "black_major_rate":      black_major_moves / bc,
        "black_avg_move_number": np.mean(black_move_numbers) if black_move_numbers else 0,

        "WhiteElo": int(game.headers.get("WhiteElo", 0)),
        "BlackElo": int(game.headers.get("BlackElo", 0)),
    }

## Step 2: Load PGN

In [ ]:
pgn_path = r"C:\Users\Admin\source\Phyton\ARJK\data.pgn"  # <-- update this

rows = []
with open(pgn_path, encoding="utf-8") as pgn:
    while True:
        game = chess.pgn.read_game(pgn)
        if game is None:
            break
        rows.append(extract_features(game))

df = pd.DataFrame(rows)
print(f"Total games loaded: {len(df)}")
df.head(3)

## Step 3: Clean & Engineer Features

In [ ]:
# Drop missing/invalid Elo
df = df[(df["WhiteElo"] > 0) & (df["BlackElo"] > 0)].copy()
df = df[(df["WhiteElo"] >= 800) & (df["WhiteElo"] <= 2900)]
df = df[(df["BlackElo"] >= 800) & (df["BlackElo"] <= 2900)]

# Encode result
result_map = {"1-0": 1, "0-1": 0, "1/2-1/2": 0.5}
df["result_encoded"] = df["result"].map(result_map).fillna(0.5)
df.drop(columns=["result"], inplace=True)

# Interaction / diff features
df["check_diff"]         = df["white_checks"]       - df["black_checks"]
df["capture_diff"]       = df["white_captures"]     - df["black_captures"]
df["castle_timing_diff"] = df["black_castle_move"]  - df["white_castle_move"]
df["piece_div_diff"]     = df["white_piece_diversity"] - df["black_piece_diversity"]
df["major_rate_diff"]    = df["white_major_rate"]   - df["black_major_rate"]
df["minor_rate_diff"]    = df["white_minor_rate"]   - df["black_minor_rate"]
df["combined_activity"]  = (df["white_captures"] + df["black_captures"]
                           + df["white_checks"]   + df["black_checks"])
df["game_complexity"]    = df["total_moves"] * (df["white_captures"] + df["black_captures"] + 1)

print(f"Games after cleaning: {len(df)}")
print(df[["WhiteElo","BlackElo"]].describe())

## Step 4: Elo Distribution Check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["WhiteElo"].hist(bins=50, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("White Elo Distribution"); axes[0].set_xlabel("Elo")
df["BlackElo"].hist(bins=50, ax=axes[1], color="coral",     edgecolor="white")
axes[1].set_title("Black Elo Distribution"); axes[1].set_xlabel("Elo")
plt.tight_layout()
plt.show()

## Step 5: Define Features & Split

**Key design choice:** We use the opponent's Elo as a feature.

This is valid because:
- We're predicting both players' Elos simultaneously from the same game
- In a real matchmaking system, you'd know one to predict the other
- This dataset is high-Elo only — behavioral stats alone have very low correlation with Elo here
- Without it, R² ≈ 0.09. With it, R² ≈ 0.56

In [ ]:
base_features = [
    "total_moves", "result_encoded",
    "white_move_count", "white_captures", "white_checks",
    "white_castled", "white_castle_move", "white_promotions",
    "white_piece_diversity", "white_capture_rate", "white_check_rate",
    "white_pawn_rate", "white_minor_rate", "white_major_rate",
    "white_avg_move_number",
    "black_move_count", "black_captures", "black_checks",
    "black_castled", "black_castle_move", "black_promotions",
    "black_piece_diversity", "black_capture_rate", "black_check_rate",
    "black_pawn_rate", "black_minor_rate", "black_major_rate",
    "black_avg_move_number",
    "check_diff", "capture_diff", "castle_timing_diff",
    "piece_div_diff", "major_rate_diff", "minor_rate_diff",
    "combined_activity", "game_complexity",
]

# Opponent Elo is the strongest single feature on this dataset
white_features = base_features + ["BlackElo"]
black_features = base_features + ["WhiteElo"]

X_white = df[white_features]
X_black = df[black_features]
y_white = df["WhiteElo"]
y_black = df["BlackElo"]

Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_white, y_white, test_size=0.2, random_state=42)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_black, y_black, test_size=0.2, random_state=42)

print(f"Train: {len(Xw_train)} games | Test: {len(Xw_test)} games")

## Step 6: Train XGBoost Models

In [ ]:
params = dict(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.05,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0,
    n_jobs=-1
)

model_white = XGBRegressor(**params)
model_black = XGBRegressor(**params)

print("Training white model...")
model_white.fit(Xw_train, yw_train)
print("Training black model...")
model_black.fit(Xb_train, yb_train)
print("Done.")

## Step 7: Evaluate

In [ ]:
yw_pred = model_white.predict(Xw_test)
yb_pred = model_black.predict(Xb_test)

print("=== White Elo ===")
print(f"  MAE:  {mean_absolute_error(yw_test, yw_pred):.1f}")
print(f"  RMSE: {np.sqrt(np.mean((yw_test - yw_pred)**2)):.1f}")
print(f"  R²:   {r2_score(yw_test, yw_pred):.4f}")

print("\n=== Black Elo ===")
print(f"  MAE:  {mean_absolute_error(yb_test, yb_pred):.1f}")
print(f"  RMSE: {np.sqrt(np.mean((yb_test - yb_pred)**2)):.1f}")
print(f"  R²:   {r2_score(yb_test, yb_pred):.4f}")

## Step 8: Sample Predictions

In [ ]:
print("White — Predicted:", yw_pred[:5].round(0))
print("White — Actual:   ", yw_test.values[:5])
print("\nBlack — Predicted:", yb_pred[:5].round(0))
print("Black — Actual:   ", yb_test.values[:5])

## Step 9: Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, actual, predicted, label, color in [
    (axes[0], yw_test, yw_pred, "White", "steelblue"),
    (axes[1], yb_test, yb_pred, "Black", "coral"),
]:
    ax.scatter(actual, predicted, alpha=0.2, s=8, color=color)
    lo = min(actual.min(), predicted.min())
    hi = max(actual.max(), predicted.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='Perfect fit')
    ax.set_xlabel("Actual Elo")
    ax.set_ylabel("Predicted Elo")
    ax.set_title(f"{label} Elo: Actual vs Predicted")
    ax.legend()
plt.tight_layout()
plt.show()

## Step 10: Feature Importance

In [ ]:
for label, model, features in [
    ("White", model_white, white_features),
    ("Black", model_black, black_features),
]:
    importance = pd.Series(model.feature_importances_, index=features).sort_values()
    importance.plot(kind="barh", figsize=(9, 7),
                    title=f"{label} Elo — Feature Importance", color="steelblue")
    plt.tight_layout()
    plt.show()